# ⚡ PowerCast — Demo Notebook
Forecast a household's electricity use for the next 24 hours with our final model, **LSTM V4**.

**Inputs:** the last 168 hours of consumption + the calendar of the next 24 hours (`is_weekend`, `hour_sin`, `hour_cos`, known in advance, no leakage).
**Output:** the next 24 hours (kW). The model is loaded by `predict.py` from `models/` (no training here).

## 0. Setup (Google Colab only)

In [ ]:
# Run this cell only on Google Colab
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/PowerCast

## 1. Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from predict import predict_next_24h, make_future_features

## 2. Load the data

In [ ]:
df = pd.read_csv("data/hourly_clean.csv", parse_dates=["datetime"])
series = df.set_index("datetime")["Global_active_power"]
test = df[df["split"] == "test"]

print("Rows:", len(df))
print("Test from", test["datetime"].min(), "to", test["datetime"].max())

## 3. Forecast one day

In [ ]:
day = pd.Timestamp("2010-08-17")

history = series[day - pd.Timedelta(hours=168) : day - pd.Timedelta(hours=1)]
actual = series[day : day + pd.Timedelta(hours=23)]
forecast = predict_next_24h(history)   # Series with dates -> calendar features are built automatically

print("History hours:", len(history))
print("Actual hours:", len(actual))
print("Forecast:", forecast.round(2))

## 4. Actual vs Forecast chart

In [ ]:
hours = range(24)
peak = forecast.argmax()
mae = abs(actual.values - forecast).mean()

plt.figure(figsize=(11, 4))
plt.plot(hours, actual.values, color="orange", marker="o", label="Actual")
plt.plot(hours, forecast, color="royalblue", marker="o", linestyle="--", label="Forecast")
plt.axvline(peak, color="royalblue", alpha=0.3, linewidth=8)

plt.title(f"PowerCast - {day.date()} | Predicted peak {peak}:00 | MAE {mae:.2f} kW")
plt.xlabel("Hour of day")
plt.ylabel("Power (kW)")
plt.xticks(hours)
plt.grid(alpha=0.3)
plt.legend()
plt.savefig(f"figures/demo_{day.date()}.png", dpi=150)
plt.show()

## 5. Backup images for the presentation

In [ ]:
def plot_day(day_text):
    day = pd.Timestamp(day_text)
    history = series[day - pd.Timedelta(hours=168) : day - pd.Timedelta(hours=1)]
    actual = series[day : day + pd.Timedelta(hours=23)]
    forecast = predict_next_24h(history)
    peak = forecast.argmax()
    mae = abs(actual.values - forecast).mean()

    plt.figure(figsize=(11, 4))
    plt.plot(range(24), actual.values, color="orange", marker="o", label="Actual")
    plt.plot(range(24), forecast, color="royalblue", marker="o", linestyle="--", label="Forecast")
    plt.axvline(peak, color="royalblue", alpha=0.3, linewidth=8)
    plt.title(f"PowerCast - {day.date()} | Predicted peak {peak}:00 | MAE {mae:.2f} kW")
    plt.xlabel("Hour of day")
    plt.ylabel("Power (kW)")
    plt.xticks(range(24))
    plt.grid(alpha=0.3)
    plt.legend()
    plt.savefig(f"figures/demo_{day.date()}.png", dpi=150)
    plt.show()

for d in ["2010-07-15", "2010-09-10", "2010-11-01"]:
    plot_day(d)

## 6. Overall result on the test period
Average error over all 148 test days. It should match the test MAE of LSTM V4 in `00_final_demo.ipynb` (**about 0.41 kW**).

In [ ]:
days = pd.date_range("2010-07-01", "2010-11-25")
errors = []
for d in days:
    history = series[d - pd.Timedelta(hours=168) : d - pd.Timedelta(hours=1)]
    actual = series[d : d + pd.Timedelta(hours=23)]
    forecast = predict_next_24h(history)
    errors.append(abs(actual.values - forecast).mean())

print("Test days:", len(errors))
print("Average daily MAE:", round(sum(errors) / len(errors), 2), "kW")

## 7. Forecast the next 24 hours (after our data ends)
Our data ends on Friday 26 Nov 2010 at 21:00. We forecast the next 24 hours (mostly Saturday) using only the last 168 hours and the calendar of the next 24 hours.
Expected result (same as `00_final_demo.ipynb`, section 7b): peak on **Saturday at 19:00**, about **2.54 kW**, total about **32.5 kWh**.

In [ ]:
last_hour = series.index[-1]
start = last_hour + pd.Timedelta(hours=1)
history = series.iloc[-168:]                       # the last 7 days we really know

future = make_future_features(start)               # 24 rows: calendar only, no consumption
future["forecast (kW)"] = predict_next_24h(history, start)
display(future.round(3))

peak_time = future["forecast (kW)"].idxmax()
print(f"Expected peak: {peak_time:%A %H:00} ({future['forecast (kW)'].max():.2f} kW) | "
      f"Expected total: {future['forecast (kW)'].sum():.1f} kWh")

plt.figure(figsize=(12, 4))
plt.plot(series.iloc[-48:].index, series.iloc[-48:].values, color="orange", marker="o", label="Actual (last 2 days)")
plt.plot(future.index, future["forecast (kW)"], color="royalblue", marker="o", linestyle="--", label="Forecast (next 24 h)")
plt.axvline(peak_time, color="royalblue", alpha=0.3, linewidth=8)
plt.title(f"PowerCast - next 24 hours | Expected peak {peak_time:%a %H}:00")
plt.ylabel("Power (kW)")
plt.grid(alpha=0.3)
plt.legend()
plt.savefig("figures/demo_next_24h.png", dpi=150)
plt.show()